In [ ]:
# Demo
# MNIST dataset- MNIST dataset  + Goal: estimating summation of handwritten digit images  + Input: a set of 28x28 images  + Output: summation of given handwritten digit images- Defining an CNN+LSTM for MNIST dataset- Training and evaluating an CNN+LSTM

# Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# PyTorch CNN+LSTM modelfrom torch.optim import SGD

# For loading MNISTfrom torchvision import datasets, transforms

# Set random seed for reproducibilitytorch.manual_seed(42)
np.random.seed(42)

# (1) Data

In [ ]:
# Load MNIST datatransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders - the CNN+LSTM expects sequences of imagesbatch_size = 128
n_images_per_set = 8  # number of images per summation set

# Custom collate function to group images into setsdef collate_fn(batch):
    images, labels = zip(*batch)
    # images: list of (1, 28, 28) tensors
    # Stack and reshape to (batch*n_images, 1, 28, 28)
    all_images = torch.stack([img for img_set in images for img in img_set])
    # Reshape to (batch, n_images, 1, 28, 28)
    batch_size = len(images)
    all_images = all_images.view(batch_size, n_images_per_set, 1, 28, 28)
    labels = torch.stack(labels)
    return all_images, labels

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# Show data shapefor images, labels in train_loader:
    print('x_train shape:', images.shape)
    print(images.shape[0], 'train samples')
    break

# (2) Model definition

In [ ]:
# CNN+LSTM model for MNIST summationclass CNN_LSTM(nn.Module):
    def __init__(self):
        super(CNN_LSTM, self).__init__()
        # CNN feature extractor        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 7x7
        )
        # After CNN, each image gives 64*7*7 = 3136 features        # LSTM processing sequence of 8 CNN features        self.lstm = nn.LSTM(input_size=64*7*7, hidden_size=128, num_layers=1, batch_first=True)
        # Final prediction layer        self.fc = nn.Linear(128, 1)  # Regression: predict summation

    def forward(self, x):
        # x shape: (batch, 8, 1, 28, 28) - 8 images of 28x28
        batch_size = x.size(0)

        # Process each image through CNN        # Reshape to (batch*8, 1, 28, 28) for CNN        cnn_input = x.view(batch_size * 8, 1, 28, 28)
        cnn_features = self.conv(cnn_input)
        # Flatten CNN features        cnn_features = cnn_features.view(-1, 64 * 7 * 7)
        # Reshape for LSTM: (batch, 8, 64*7*7)        lstm_input = cnn_features.view(batch_size, 8, 64 * 7 * 7)
        # LSTM forward        lstm_out, (hn, cn) = self.lstm(lstm_input)
        # Use the last timestep's output        last_lstm_out = lstm_out[:, -1, :]
        # Final fully connected layer        output = self.fc(last_lstm_out)
        return output

# Initialize model, loss, and optimizerdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNN_LSTM().to(device)
criterion = nn.L1Loss()  # Mean Absolute Error for summationoptimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

In [ ]:
# Training loopnum_epochs = 30
batch_size = 128

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass        outputs = model(images)
        loss = criterion(outputs, labels.float().unsqueeze(1))

        # Backward and optimize        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 100 == 0:
            print[f'Epoch [{epoch+1}/30], Step [{i+1}/{(len(train_dataset)//batch_size)}], Loss: {running_loss / 100:.4f}']
            running_loss = 0.0

    print(f'Epoch [{epoch+1}/30] completed')

# Print training timestart_time = time.time()

In [ ]:
# Test the modelmodel.eval()
with torch.no_grad():
    total_loss = 0.0
    total_samples = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels.float().unsqueeze(1))
        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

print(f'Mean absolute error of testing set: {total_loss / total_samples:.6f}')